# 04 — Valuation EDA (UAE used-car listings)

**Dataset:** `alikalwar/uae-used-car-prices-and-features-10k-listings` (10,000 real UAE used-car listings, CC-open).
Understand price/mileage/age structure and the parsed *condition* signal before training the XGBoost valuation model (notebook 05).

> Runs locally or on Kaggle. Reads the cleaned table produced by `data/prepare_tabular.py`.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Local path; on Kaggle point to the attached dataset / re-run prepare_tabular.py first.
df = pd.read_parquet("../data/processed/listings_clean.parquet")
print(df.shape)
df.head()

## 1. Target: price is heavily right-skewed → log transform

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
df["Price"].plot.hist(bins=60, ax=ax[0], title="Price (AED)")
df["log_price"].plot.hist(bins=60, ax=ax[1], title="log1p(Price)")
plt.tight_layout(); plt.show()
df[["Price","Mileage","age"]].describe()

## 2. Age & mileage vs price

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].scatter(df["age"], df["Price"], s=4, alpha=0.2); ax[0].set(xlabel="age (yrs)", ylabel="Price", title="Age vs Price")
ax[1].scatter(df["Mileage"], df["Price"], s=4, alpha=0.2); ax[1].set(xlabel="Mileage (km)", ylabel="Price", title="Mileage vs Price")
plt.tight_layout(); plt.show()

## 3. Condition signal — the CV↔price bridge
Parsed from listing text. This is what the CV damage detector's output maps onto at inference time.

In [ ]:
order = ["no_damage","minor_scratches","repainted_bumper","dented_door","engine_repaired","accident_history"]
g = df.groupby("condition")["Price"].median().reindex(order)
g.plot.bar(title="Median price by condition", figsize=(8,4)); plt.ylabel("median AED"); plt.tight_layout(); plt.show()
df.groupby("condition")["Price"].agg(["count","median"]).reindex(order)

## 4. Brand-level price tiers (top 12 by volume)

In [ ]:
top = df["Make"].value_counts().head(12).index
df[df["Make"].isin(top)].groupby("Make")["Price"].median().sort_values().plot.barh(figsize=(8,5), title="Median price by make")
plt.xlabel("median AED"); plt.tight_layout(); plt.show()

## 5. Numeric correlations with log_price

In [ ]:
num = df[["age","Mileage","mileage_per_year","Cylinders","condition_severity","log_price"]]
corr = num.corr(numeric_only=True)["log_price"].sort_values()
print(corr)

## Takeaways for modelling (notebook 05)
- Log-price target (right-skew handled).
- `age`, `Mileage`, `condition_severity`, `Cylinders`, `Make`/`Model` are the strongest levers → gradient boosting with categorical support fits well.
- Condition classes are near-balanced (~1600 each) and monotonically depress price → SHAP directional checks should confirm this.